# LingoPrism – Huấn luyện YOLOv10 (78 lớp)

**Cách chạy**
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**. Khi được hỏi quyền Google Drive thì bấm cho phép.
3. Giữ tab mở (Colab miễn phí ngắt khi bỏ tab quá lâu). Mỗi epoch vài phút sau khi ảnh được thu nhỏ (bản v3) – Colab miễn phí sẽ ngắt, cứ *Run all* lại.
4. Ô cuối tự tải về `lingoprism-yolov10n-v3.zip` → gửi file đó lại để đăng ký model vào server.

**Nếu bị ngắt giữa chừng:** chỉ cần *Run all* lại – checkpoint nằm trên Drive (`MyDrive/lingoprism_runs`), notebook sẽ train tiếp chứ không train lại từ đầu.


**Thời gian (Colab miễn phí, GPU T4):** dựng dataset 150 ảnh/lớp ~30–40 phút (lần sau giải nén từ Drive vài phút), train tối đa 4 giờ (tự dừng), đánh giá + export ~10 phút. Tải file zip cuối notebook về rồi gửi lại để so với model đang chạy trên tập test 2 542 ảnh (`AI/scripts/eval_testset.py`).

In [ ]:
# 1. Kiểm tra GPU, lấy mã nguồn, cài thư viện
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
if not os.path.exists('/content/LingoPrism'):
    !git clone -q https://github.com/GPTii/LingoPrism.git /content/LingoPrism
else:
    !git -C /content/LingoPrism pull -q
%cd /content/LingoPrism
!pip install -q ultralytics fiftyone pyyaml onnx onnxslim onnxruntime
import torch
assert torch.cuda.is_available(), 'Chưa bật GPU: Runtime → Change runtime type → T4 GPU'


In [ ]:
# 2. Cấu hình – có thể chỉnh
import os
os.chdir('/content/LingoPrism')  # Colab có thể khởi động lại phiên sau ô 1 → mất %cd và import
# v1 (150 ảnh/lớp, 50 epoch): mAP50 = 0,33 – mAP vẫn đang tăng khi dừng (underfitting), kém COCO gốc trên các lớp chung.
# v2: nhiều ảnh hơn + train lâu hơn – vẫn học trên dữ liệu thiếu nhãn (gối trong ảnh COCO, người trong ảnh Open Images
#     không được gắn nhãn → model bị dạy là "nền").
# v3: lọc nhãn rác (crowd / group-of / tranh vẽ) + gán nhãn bù bằng model lớn đã train sẵn (PSEUDO) cho tập train.
#     So sánh với model pretrained đang chạy trên server: AI/scripts/eval_photos.py trên bộ ảnh tự chụp.
# v4 (nhanh – mặc định): KHÔNG train từ đầu nữa. Fine-tune từ YOLOv8s Open Images V7 (model đang chạy trên server, đã
#     biết sẵn cả 78 lớp): giữ nguyên backbone (FREEZE), ít ảnh hơn, giới hạn cứng HOURS giờ → xong trong 1 phiên Colab.
#     Train từ đầu yolov10n với 400 ảnh/lớp mất 25–40 giờ trên GPU T4 + 2 CPU của Colab miễn phí (bị ngắt trước khi xong).
PER_CLASS = 150          # ảnh tối đa mỗi lớp mỗi nguồn. Nếu Drive đã có lingoprism_ds_v3_400_pseudo.tar thì đặt 400 để dùng lại
EPOCHS    = 30           # tối đa; HOURS dừng trước nếu hết giờ, PATIENCE dừng khi mAP không còn tăng
HOURS     = 4            # giới hạn thời gian train (giờ) – lịch learning rate tự co theo
PATIENCE  = 8
MODEL     = 'yolov8s-oiv7.pt'  # điểm xuất phát đã train sẵn (Ultralytics tự tải); 'yolov10n.pt' = train từ COCO (rất lâu)
FREEZE    = 10           # 10 lớp backbone giữ nguyên (nhanh hơn, không quên Open Images); 0 = train hết
NAME      = 'lingoprism-yolov8s-ft'
PSEUDO    = True         # gán nhãn bù cho tập train (~15 phút với 150 ảnh/lớp)

from google.colab import drive
drive.mount('/content/drive')
DS   = '/content/lingoprism_ds_v3'                 # dataset ở ổ Colab (nhanh, không tốn Drive); đổi tên theo phiên bản
RUNS = '/content/drive/MyDrive/lingoprism_runs'    # checkpoint + kết quả ở Drive (giữ khi bị ngắt)
os.makedirs(RUNS, exist_ok=True)


In [ ]:
# 3. Dataset từ COCO-2017 + Open Images V7
#    - Lần đầu: tải + lọc + gán nhãn bù (~1 giờ), rồi lưu 1 file .tar lên Drive
#    - Bị ngắt / phiên mới: giải nén lại từ Drive (vài phút) thay vì tải lại từ đầu
#    - Ảnh được thu nhỏ về cạnh dài 640px → DataLoader trên 2 CPU của Colab không còn nghẽn (nhanh hơn nhiều)
TAR = f'{RUNS}/lingoprism_ds_v3_{PER_CLASS}{"_pseudo" if PSEUDO else ""}.tar'
FLAGS = '--pseudo-label' if PSEUDO else ''
if not os.path.exists(f'{DS}/data.yaml'):
    if os.path.exists(TAR):
        print('Giải nén dataset từ Drive...')
        !tar -xf {TAR} -C /content
    else:
        !python /content/LingoPrism/AI/scripts/build_dataset.py --per-class {PER_CLASS} --out {DS} {FLAGS}
!python /content/LingoPrism/AI/scripts/build_dataset.py --shrink-only --out {DS}
!rm -f {DS}/labels/*.cache
if not os.path.exists(TAR):
    print('Lưu dataset lên Drive...')
    !tar -cf {TAR} -C /content {os.path.basename(DS)}
print('Dataset sẵn sàng:', DS)

In [ ]:
# 3b. Gộp báo lỗi của người dùng (nếu có): Admin → Nhận diện YOLO → Báo lỗi → "Xuất dữ liệu train"
#     → tải file zip lên Drive với tên MyDrive/lingoprism_feedback.zip. Model sẽ học đúng các ca đã bị báo sai.
FEEDBACK = '/content/drive/MyDrive/lingoprism_feedback.zip'
if os.path.exists(FEEDBACK):
    !python /content/LingoPrism/AI/scripts/merge_feedback.py --zip {FEEDBACK} --data {DS}/data.yaml
else:
    print('Không có file báo lỗi – bỏ qua')

In [ ]:
# 4. Train (tự train tiếp nếu có checkpoint trên Drive) + đánh giá trên tập test + export ONNX & TFLite
#    Giữ tab Colab mở (Colab miễn phí ngắt phiên khi để yên ~90 phút). Bị ngắt: chạy lại ô 1 → 2 → 3 → 4, tự train tiếp.
!python /content/LingoPrism/AI/scripts/train_eval_export.py --data {DS}/data.yaml --model {MODEL} --epochs {EPOCHS} \
    --hours {HOURS} --patience {PATIENCE} --freeze {FREEZE} --project {RUNS} --name {NAME}


In [ ]:
# 5. Kết quả
import json
from IPython.display import Image, display
run = f'{RUNS}/{NAME}'
m = json.load(open(f'{run}/report/metrics.json'))
print(f"Precision {m['precision']:.3f} | Recall {m['recall']:.3f} | mAP50 {m['map50']:.3f} | mAP50-95 {m['map50_95']:.3f}")
display(Image(f'{run}/results.png', width=900))
display(Image(f'{run}/test/confusion_matrix_normalized.png', width=900))


In [ ]:
# 6. Nén & tải về (gửi file zip này lại cho server)
import glob, shutil, zipfile
from google.colab import files
out = f'/content/{NAME}.zip'
keep = ['model_card.json', 'results.csv', 'results.png', 'args.yaml']
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in keep:
        if os.path.exists(f'{run}/{f}'):
            z.write(f'{run}/{f}', f)
    for f in glob.glob(f'{run}/weights/best*') + glob.glob(f'{run}/weights/best_saved_model/*float16.tflite'):
        if os.path.isfile(f):
            z.write(f, 'weights/' + os.path.basename(f))
    for folder in ('report', 'test'):
        for f in glob.glob(f'{run}/{folder}/*'):
            if os.path.isfile(f):
                z.write(f, f'{folder}/' + os.path.basename(f))
print(out, round(os.path.getsize(out) / 1e6, 1), 'MB')
files.download(out)
